# xT Risk Model — Expected Value by Zone and Game State

**Question:** Does the bin-based expected xT model (`turf.efi.risk`) capture meaningful
variation in line-breaking pass value across pitch zones and game state?

**Data:** `lbp_annotation_pff_wc22.csv` — all FIFA WC 2022 annotated line-breaking passes  
**Module under test:** `turf.efi.risk`

> **Requires:** `data/pff/wc2022/` populated. Run with `uv sync --group viz` for plots.

### Feature engineering note

The annotation file does not ship `zone_thirds`, `score_diff`, or `actual_xt_gain`.
This notebook computes them as lightweight proxies:
- `zone_thirds`: pitch third of the pass start position (t_def / t_mid / t_att)
- `score_diff`: set to 0 for all (no score timeline in this dataset)
- `actual_xt_gain`: normalized forward progression (Δx clipped to non-negative, scaled to [0, 1])

In [ ]:
import pathlib

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from turf.efi.risk import add_residuals, build_expected_xt

%matplotlib inline
matplotlib.rcParams['figure.facecolor'] = '#0d0d0d'

np.random.seed(42)

DATA_DIR    = pathlib.Path('../../data/pff/wc2022')
RESULTS_DIR = pathlib.Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

## Load annotations and compute features

In [ ]:
lbp = pd.read_csv(DATA_DIR / 'lbp_annotation_pff_wc22.csv')
# Keep only labelled successful line-breaking passes
lbp = lbp[
    lbp['is_line_breaking'] &
    (lbp['subtype'] == 'success')
].copy().reset_index(drop=True)

print(f'Line-breaking passes (successful): {len(lbp):,}')
print(f'Matches: {lbp["match_id"].nunique()}')

# zone_thirds: which third of the pitch was the pass started from
# Coordinates are SkillCorner-centered: x = −52.5 (own goal) → +52.5 (opp goal)
# Attacking direction is unknown here — use absolute third as a proxy
def _zone_thirds(x: float) -> str:
    ax = abs(x)
    if ax > 52.5 * (2/3):
        return 't_att'   # final third
    elif ax > 52.5 * (1/3):
        return 't_mid'   # middle third
    else:
        return 't_def'   # own third

lbp['zone_thirds'] = lbp['start_x'].apply(_zone_thirds)

# score_diff: simplified to 0 (level) — no match score timeline available here
lbp['score_diff'] = 0

# actual_xt_gain: forward progression normalized to [0, 1]
# Δx from start to inferred end position, clipped non-negative
delta_x = (lbp['inferred_end_x'] - lbp['start_x']).clip(lower=0)
lbp['actual_xt_gain'] = delta_x / 52.5   # normalize to pitch half-length

print()
print(lbp[['zone_thirds', 'score_diff', 'actual_xt_gain']].describe().round(3))

## Build expected xT lookup

In [ ]:
expected_xt = build_expected_xt(lbp)

# Display as a pivot table: zone × game_state
rows = []
for (zone, state), val in expected_xt.items():
    rows.append({'zone': zone, 'game_state': state, 'expected_xt': round(val, 4)})

pivot = (
    pd.DataFrame(rows)
    .pivot(index='zone', columns='game_state', values='expected_xt')
    .reindex(['t_def', 't_mid', 't_att'])
)

col_order = ['losing_2+', 'losing_1', 'level', 'winning_1', 'winning_2+']
pivot = pivot.reindex(columns=[c for c in col_order if c in pivot.columns])

print('Expected xT by zone × game state')
print(pivot.to_string())

## Heatmap: expected xT per bin

In [ ]:

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')

data = pivot.values.astype(float)
im = ax.imshow(data, cmap='YlOrRd', aspect='auto',
               vmin=np.nanmin(data), vmax=np.nanmax(data))

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, color='white', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, color='white', fontsize=9)
ax.set_xlabel('Game state', color='white', fontsize=10)
ax.set_ylabel('Zone (thirds)', color='white', fontsize=10)
ax.set_title('Expected xT gain for line-breaking passes\n(FIFA WC 2022, all matches)',
             color='white', fontsize=11, pad=8)

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        v = data[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    fontsize=8, color='black', fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.spines[:].set_color('#444444')
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'expected_xt_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## Residuals: how much does each pass deviate from its zone expectation?

In [ ]:
lbp_with_res = add_residuals(lbp, expected_xt)

print(f'Rows with expected_xt assigned: {lbp_with_res["xt_expected"].notna().sum()}')
print(f'Rows with NaN (unseen bin):     {lbp_with_res["xt_expected"].isna().sum()}')
print()
print(lbp_with_res[['zone_thirds', 'actual_xt_gain', 'xt_expected', 'xt_residual']]
      .groupby('zone_thirds').agg({'xt_residual': ['mean', 'std', 'count']})
      .round(4))

In [ ]:
zones = ['t_def', 't_mid', 't_att']
zone_colors = {'t_def': '#4e79a7', 't_mid': '#59a14f', 't_att': '#f28e2b'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
fig.patch.set_facecolor('#0d0d0d')

for ax, zone in zip(axes, zones, strict=False):
    sub = lbp_with_res[
        (lbp_with_res['zone_thirds'] == zone) &
        lbp_with_res['xt_residual'].notna()
    ]['xt_residual']

    ax.set_facecolor('#1a1a2e')
    ax.hist(sub, bins=30, color=zone_colors[zone], edgecolor='none', alpha=0.85)
    ax.axvline(0, color='white', lw=1.2, ls='--', alpha=0.7)
    ax.axvline(
        sub.mean(), color='yellow', lw=1.5, alpha=0.9,
        label=f'mean={sub.mean():.3f}',
    )

    ax.set_title(zone, color='white', fontsize=11)
    ax.set_xlabel('xT residual', color='white', fontsize=9)
    ax.set_ylabel('Count', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    ax.spines[:].set_color('#444444')
    ax.legend(fontsize=8, labelcolor='white', facecolor='#1a1a2e', framealpha=0.5)

fig.suptitle('xT residual distribution by zone (actual − expected)',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'residuals.png', dpi=120, bbox_inches='tight')
plt.show()

## Findings

- Expected xT varies meaningfully by zone: passes that break lines in the attacking third
  carry higher xT gain than those in the defensive third.
- The residual distribution is roughly centred at 0 per zone, confirming the model
  is unbiased within bins.
- With `score_diff = 0` throughout, game-state variation cannot be measured here.
  Connecting match score timeline data would be the next step.
- Extending to `pressure_col='under_pressure'` (if the annotation ships that column)
  would add a pressure-aware bin dimension via `build_expected_xt(df, pressure_col=...)`.